In [2]:
import math
from typing import cast

import gymnasium as gym
import torch
from gymnasium.spaces import Box, Discrete
from model import DQN, ReplayBuffer, Transition
from torch import nn, optim


In [3]:
BATCH_SIZE = 128
HIDDEN_SIZE = 128
GAMMA = 0.99
EPS_START = 0.9
EPS_END = 0.01
EPS_DECAY = 2500
TAU = 0.005
LR = 3e-4

In [4]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

In [5]:
env = gym.make("CartPole-v1", render_mode="human")

obs_dim = cast(Box, env.observation_space).shape[0]
action_dim = cast(Discrete, env.action_space).n

In [6]:
policy_net = DQN(obs_dim, action_dim, HIDDEN_SIZE)
target_net = DQN(obs_dim, action_dim, HIDDEN_SIZE)
optimizer = optim.AdamW(policy_net.parameters(), lr=LR, amsgrad=True)
memory = ReplayBuffer(10000)

In [ ]:
def optimize_model():
    if len(memory) < BATCH_SIZE:
        return
    transitions = memory.sample(BATCH_SIZE)
    batch = Transition(*zip(*transitions))
    non_final_mask = torch.tensor(
        tuple(s is not None for s in batch.next_state),
        device=device,
        dtype=torch.bool,
    )
    non_final_next_states = torch.cat([s for s in batch.next_state if s is not None])
    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    reward_batch = torch.cat(batch.reward)

    state_action_values = policy_net(state_batch).gather(1, action_batch)
    next_state_values = torch.zeros(BATCH_SIZE, device=device)

    with torch.no_grad():
        next_state_values[non_final_mask] = (
            target_net(non_final_next_states).max(1).values
        )
    expected_state_action_values = (next_state_values * GAMMA) + reward_batch

    criterion = nn.SmoothL1Loss()
    loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)
    optimizer.step()

: 

In [ ]:
obs, info = env.reset()
total_reward = 0

for step in range(100000):
    state = torch.tensor(obs, dtype=torch.float32)

    eps_threshold = EPS_END + (EPS_START - EPS_END) * math.exp(-1.0 * step / EPS_DECAY)
    if torch.rand(1).item() < eps_threshold:
        action = torch.tensor(
            [[env.action_space.sample()]], device=device, dtype=torch.long
        )
    else:
        action = torch.argmax(policy_net(state)).view(-1, 1)

    # print(action.item())
    observation, reward, terminated, truncated, info = env.step(action.item())
    reward = torch.tensor([reward], device=device)
    done = terminated or truncated

    if terminated:
        next_state = None
    else:
        next_state = torch.tensor(observation, dtype=torch.float32)

    memory.push(state, action, next_state, reward)

    optimize_model()

    if done:
        obs, info = env.reset()
        print("Total reward:", total_reward)
        total_reward = 0

env.close()

Total reward: 0
Total reward: 0
Total reward: 0
Total reward: 0
Total reward: 0
Total reward: 0
